# Классификация новостей Lenta.ru с усредненными эмбеддингами

1. Загружаю корпус `lenta-ru-news`, очищаю данные и строю стратифицированные выборки `60/20/20`.
2. Фиксирую единое текстовое поле `title + text`, а дальше варьирую только семейство эмбеддингов.
3. Обучаю собственные `Word2Vec`эмбеддинги через `gensim`, отдельно проверяю их внутреннее качество через `most_similar` и `doesnt_match`.
4. Загружаю два предобученных семейства: `navec` и `RusVectores`, затем обучаю `LogisticRegression` на усредненных эмбеддингах.
5. Для лучшего семейства дополнительно проверяю `tf-idf`взвешенное усреднение, потому что простое среднее одинаково учитывает и тематические слова, и почти бесполезные служебные токены.
6. Во всех сравнениях слежу за `accuracy`, `macro F1` и `weighted F1`, но модель выбираю по `macro F1`: распределение тем в Lenta.ru заметно неравномерное, и именно `macro F1` лучше показывает, насколько модель держит редкие классы, а не только самые массовые.

Импорты:


In [1]:
import contextlib
import functools
import io
import os
import pickle
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import KeyedVectors, Word2Vec
from IPython.display import display
from navec import Navec
from pymorphy3 import MorphAnalyzer
from razdel import tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Разделение датасета на train / validation / test

Загружаю корпус и делаю ту же подвыборку, что и в первом ДЗ. Убираю пустые записи, оставляю `title`, `text`, `topic`, беру стратифицированный срез на `100 000` текстов.

Настройки для загрузки и фильтрации данных:


In [2]:
RANDOM_STATE = 42
SAMPLE_SIZE = 100_000
MIN_TOPIC_COUNT = 5

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"
RAW_DATA_PATH = DATA_DIR / "lenta-ru-news.csv.gz"
RAW_DATA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
SPLIT_CACHE_PATH = CACHE_DIR / "hw2_splits_100k.pkl"

`MIN_TOPIC_COUNT = 5` гарантирует, что при разбиении `60/20/20` в каждом сплите у каждой темы окажется хотя бы несколько примеров.

Регулярные выражения для нормализации и маппинг POS тегов (нужен для формата словаря `RusVectores`: `lemma_POS`):


In [3]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b", re.UNICODE)
SPACE_RE = re.compile(r"\s+", re.UNICODE)
WORD_RE = re.compile(r"^[a-zа-яё]+$", re.IGNORECASE)
MARKER_RE = re.compile(r"^__(?:url|num)__$")

MORPH = MorphAnalyzer()

POS_MAP = {
    "NOUN": "NOUN",
    "ADJF": "ADJ",
    "ADJS": "ADJ",
    "COMP": "ADJ",
    "VERB": "VERB",
    "INFN": "VERB",
    "PRTF": "VERB",
    "PRTS": "VERB",
    "GRND": "VERB",
    "NUMR": "NUM",
    "ADVB": "ADV",
    "NPRO": "PRON",
    "PRED": "ADV",
    "PREP": "ADP",
    "CONJ": "CCONJ",
    "PRCL": "PART",
    "INTJ": "INTJ",
}

Загрузка и подготовка данных:


In [4]:
def ensure_download(path: Path, url: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path


def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, compression="gzip", usecols=["title", "text", "topic"])
    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    df["topic"] = df["topic"].fillna("").astype(str)
    df = df.loc[df["text"].str.strip().ne("") & df["topic"].str.strip().ne("")].copy()
    df = df.drop_duplicates(subset=["title", "text", "topic"]).reset_index(drop=True)
    return df


raw_path = ensure_download(RAW_DATA_PATH, RAW_DATA_URL)
dataset = load_dataset(raw_path)
dataset.shape

(738949, 3)

Функции токенизации. Для каждого текста строю три набора токенов, потому что каждая модель эмбеддингов ожидает свой формат:


In [5]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" __url__ ", text)
    text = NUM_RE.sub(" __num__ ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def tokenize_text(text: str) -> list[str]:
    tokens = []
    for token in tokenize(normalize_text(text)):
        value = token.text.strip()
        if not value:
            continue
        if MARKER_RE.fullmatch(value) or WORD_RE.fullmatch(value):
            tokens.append(value)
    return tokens


@functools.cache
def analyze_token(token: str) -> tuple[str, str | None]:
    if MARKER_RE.fullmatch(token):
        return token, None
    parsed = MORPH.parse(token)[0]
    return parsed.normal_form, POS_MAP.get(parsed.tag.POS)


def make_lemma_tokens(tokens: list[str]) -> list[str]:
    return [analyze_token(token)[0] for token in tokens]


def make_rusvectores_tokens(tokens: list[str]) -> list[str]:
    rv_tokens = []
    for token in tokens:
        lemma, pos = analyze_token(token)
        if pos is None:
            continue
        rv_tokens.append(f"{lemma}_{pos}")
    return rv_tokens

Сборка сплитов (или загрузка из кэша):


In [ ]:
def build_or_load_splits() -> dict[str, pd.DataFrame]:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if SPLIT_CACHE_PATH.exists():
        with SPLIT_CACHE_PATH.open("rb") as handle:
            return pickle.load(handle)

    topic_counts = dataset["topic"].value_counts()
    filtered = dataset.loc[dataset["topic"].isin(topic_counts[topic_counts >= MIN_TOPIC_COUNT].index)].reset_index(
        drop=True
    )

    sample_fraction = SAMPLE_SIZE / len(filtered)
    min_count_for_sample = int(np.ceil(MIN_TOPIC_COUNT / sample_fraction))
    sample_topic_counts = filtered["topic"].value_counts()
    filtered = filtered.loc[
        filtered["topic"].isin(sample_topic_counts[sample_topic_counts >= min_count_for_sample].index)
    ].reset_index(drop=True)

    sample_df, _ = train_test_split(
        filtered,
        train_size=SAMPLE_SIZE,
        stratify=filtered["topic"],
        random_state=RANDOM_STATE,
    )
    sample_df = sample_df.reset_index(drop=True)
    sample_df["combined_text"] = (sample_df["title"].str.strip() + " " + sample_df["text"].str.strip()).str.strip()

    base_tokens = sample_df["combined_text"].map(tokenize_text)
    sample_df["simple_tokens"] = base_tokens
    sample_df["lemma_tokens"] = base_tokens.map(make_lemma_tokens)
    sample_df["rv_tokens"] = base_tokens.map(make_rusvectores_tokens)

    train_df, holdout_df = train_test_split(
        sample_df,
        test_size=0.4,
        stratify=sample_df["topic"],
        random_state=RANDOM_STATE,
    )
    valid_df, test_df = train_test_split(
        holdout_df,
        test_size=0.5,
        stratify=holdout_df["topic"],
        random_state=RANDOM_STATE,
    )

    splits = {
        "train": train_df.reset_index(drop=True),
        "valid": valid_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }
    with SPLIT_CACHE_PATH.open("wb") as handle:
        pickle.dump(splits, handle)
    return splits


splits = build_or_load_splits()
train_df = splits["train"]
valid_df = splits["valid"]
test_df = splits["test"]

split_stats = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "topics": train_df["topic"].nunique()},
        {"split": "valid", "rows": len(valid_df), "topics": valid_df["topic"].nunique()},
        {"split": "test", "rows": len(test_df), "topics": test_df["topic"].nunique()},
    ]
)
display(split_stats)

topic_distribution = (
    pd.concat(
        [
            train_df["topic"].value_counts(normalize=True).rename("train"),
            valid_df["topic"].value_counts(normalize=True).rename("valid"),
            test_df["topic"].value_counts(normalize=True).rename("test"),
        ],
        axis=1,
    )
    .fillna(0.0)
    .sort_index()
)
display(topic_distribution.head(10))

,split,rows,topics
0,train,60000,19
1,valid,20000,19
2,test,20000,19


,train,valid,test
topic,,,
69-я параллель,0.001717,0.00175,0.00170
Библиотека,0.000083,0.00010,0.00010
Бизнес,0.010017,0.01000,0.01000
Бывший СССР,0.072267,0.07230,0.07225
Дом,0.029417,0.02940,0.02940
Из жизни,0.037367,0.03735,0.03735
Интернет и СМИ,0.060433,0.06045,0.06045
Крым,0.000900,0.00090,0.00090
Культпросвет,0.000467,0.00045,0.00045


По таблице видно, что разбиение получилось корректным: размеры частей ровно `60 000 / 20 000 / 20 000`, а все 19 тематик присутствуют в каждом сплите. Нормированные частоты по темам между `train`, `valid` и `test` почти совпадают, значит стратификация отработала так, как и должна.

Для дальнейшей работы сразу храню три представления текста. `simple_tokens` нужны для `Navec`, потому что эта модель обучалась на новостях и отлично покрывает обычные нижнерегистровые формы. `lemma_tokens` использую для собственной `Word2Vec`: лемматизация уменьшает словарь, и на новостях это обычно помогает устойчивее обучить вектора на ограниченном объеме данных. Для `RusVectores` отдельно строю токены вида `lemma_POS`, потому что именно в таком формате у модели записан словарь.

## 2. Обучение `Word2Vec` с помощью `gensim`


Гиперпараметры `Word2Vec`:


In [7]:
W2V_PARAMS = {
    "vector_size": 300,
    "window": 7,
    "min_count": 5,
    "sg": 1,
    "negative": 10,
    "epochs": 15,
    "sample": 1e-5,
    "workers": (os.cpu_count() or 1),
    "seed": RANDOM_STATE,
}

`vector_size=300` дает модели достаточно ёмкости и совпадает по размерности с предобученными русскоязычными моделями.

`window=7` чуть шире стандартного. Для тематической классификации полезно ловить не только ближайшие слова, но и более далеких тематических соседей по предложению.

`sg=1` (skip-gram) обычно лучше работает с редкими словами. В новостях редкие имена и организации часто несут основной тематический сигнал.

`min_count=5` убирает совсем редкий шум. 

`negative=10` и `epochs=15` дают модели достаточно итераций на корпусе из 60 тысяч документов. 

`sample=1e-5` отбрасываем частотные слова вроде предлогов, чтобы модель больше внимания уделяла содержательной лексике.


Обучение модели:


In [8]:
train_sentences = train_df["lemma_tokens"].tolist()

with contextlib.redirect_stderr(io.StringIO()):
    w2v_model = Word2Vec(sentences=train_sentences, **W2V_PARAMS)

print("Word2Vec vocab size:", len(w2v_model.wv))
print("Vector size:", w2v_model.vector_size)

Word2Vec vocab size: 48921
Vector size: 300


### 2.1 Проверка качества эмбеддингов

Проверяю через `most_similar` и `doesnt_match`.

In [9]:
intrinsic_rows = []
for query in ["россия", "футбол", "нефть"]:
    if query in w2v_model.wv:
        intrinsic_rows.append(
            {
                "query": query,
                "most_similar": ", ".join(word for word, _ in w2v_model.wv.most_similar(query, topn=5)),
            }
        )

intrinsic_df = pd.DataFrame(intrinsic_rows)
display(intrinsic_df)

doesnt_match_examples = []
for words in [
    ["футбол", "хоккей", "теннис", "президент"],
    ["доллар", "евро", "рубль", "тренер"],
]:
    present_words = [word for word in words if word in w2v_model.wv]
    if len(present_words) >= 3:
        doesnt_match_examples.append(
            {
                "words": ", ".join(present_words),
                "odd_one_out": w2v_model.wv.doesnt_match(present_words),
            }
        )

display(pd.DataFrame(doesnt_match_examples))

,query,most_similar
0,россия,"российский, рф, владимир, по, страна"
1,футбол,"сборная, футбольный, чемпионат, футболист, матч"
2,нефть,"баррель, нефтяной, бижан, opec, eia"


,words,odd_one_out
0,"футбол, хоккей, теннис, президент",президент
1,"доллар, евро, рубль, тренер",тренер


Выглядит верно

## 3. Загрузка предобученных эмбеддингов `Navec` и `RusVectores`

Для `Navec` подаю `simple_tokens`. У `RusVectores` словарь хранится в виде `lemma_POS`, поэтому отдельно строю токены с POS тегами, иначе покрытие будет искусственно заниженным.

Для `RusVectores` загружаю полный словарь модели `ruwikiruscorpora_upos_cbow_300_10_2021`.


Пути к предобученным моделям:


In [10]:
NAVEC_PATH = DATA_DIR / "embeddings" / "navec_news_v1_1B_250K_300d_100q.tar"
NAVEC_URL = "https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar"
RUSVECTORES_MODEL_PATH = DATA_DIR / "embeddings" / "ruwikiruscorpora_upos_cbow_300_10_2021.model"
RUSVECTORES_VECTORS_PATH = DATA_DIR / "embeddings" / "ruwikiruscorpora_upos_cbow_300_10_2021.model.vectors.npy"
RUSVECTORES_MODEL_URL = "https://rusvectores.org/static/models/ruwikiruscorpora_upos_cbow_300_10_2021.model"
RUSVECTORES_VECTORS_URL = (
    "https://rusvectores.org/static/models/ruwikiruscorpora_upos_cbow_300_10_2021.model.vectors.npy"
)

Загрузка:


In [11]:
navec = Navec.load(str(ensure_download(NAVEC_PATH, NAVEC_URL)))
ensure_download(RUSVECTORES_MODEL_PATH, RUSVECTORES_MODEL_URL)
ensure_download(RUSVECTORES_VECTORS_PATH, RUSVECTORES_VECTORS_URL)
rusvectores = KeyedVectors.load(str(RUSVECTORES_MODEL_PATH))

print("Navec dim:", navec.pq.dim)
print("RusVectores vocab:", len(rusvectores))
print("RusVectores dim:", rusvectores.vector_size)

Navec dim: 300
RusVectores vocab: 249333
RusVectores dim: 300


## 4. `LogisticRegression` с тремя вариантами векторизации

Каждый документ представляю одним вектором: среднее эмбеддингов слов, попавших в словарь модели. Если токена в словаре нет, пропускаю его. Если в документе не нашлось ни одного знакомого слова, возвращаю нулевой вектор.


Параметры `LogisticRegression` для каждого набора эмбеддингов подбирал вручную эмпирически:


In [12]:
W2V_LOGREG_PARAMS = {"C": 2.0, "class_weight": None, "with_scaler": True}
NAVEC_LOGREG_PARAMS = {"C": 4.0, "class_weight": "balanced", "with_scaler": False}
RUSVECTORES_LOGREG_PARAMS = {"C": 0.25, "class_weight": None, "with_scaler": True}

Для `Word2Vec` и `RusVectores` добавляю `StandardScaler`, потому что у них компоненты векторов заметно по-разному масштабированы. Для `Navec` скейлер не использую: квантизованные вектора и так достаточно устойчивы, а на validation без скейлера результат оказался лучше.


Общие функции для построения эмбеддингов и оценки:


In [13]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def build_dense_pipeline(*, C: float, class_weight, with_scaler: bool) -> Pipeline:
    steps = []
    if with_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(
        (
            "clf",
            LogisticRegression(
                C=C,
                class_weight=class_weight,
                solver="lbfgs",
                max_iter=2_000,
                random_state=RANDOM_STATE,
            ),
        )
    )
    return Pipeline(steps)


def make_mean_embeddings(token_lists: list[list[str]], model, dim: int):
    matrix = np.zeros((len(token_lists), dim), dtype=np.float32)
    covered_tokens = 0
    total_tokens = 0
    covered_docs = 0

    for row_id, tokens in enumerate(token_lists):
        vectors = []
        for token in tokens:
            total_tokens += 1
            if token in model:
                covered_tokens += 1
                vectors.append(model[token])
        if vectors:
            covered_docs += 1
            matrix[row_id] = np.mean(vectors, axis=0)

    stats = {
        "token_coverage": covered_tokens / max(total_tokens, 1),
        "document_coverage": covered_docs / max(len(token_lists), 1),
    }
    return matrix, stats


def fit_eval_dense(train_X, train_y, valid_X, valid_y, test_X, test_y, params):
    valid_model = build_dense_pipeline(**params)
    valid_model.fit(train_X, train_y)
    valid_metrics = compute_metrics(valid_y, valid_model.predict(valid_X))

    train_valid_X = np.vstack([train_X, valid_X])
    train_valid_y = pd.concat([train_y, valid_y], ignore_index=True)
    test_model = build_dense_pipeline(**params)
    test_model.fit(train_valid_X, train_valid_y)
    test_metrics = compute_metrics(test_y, test_model.predict(test_X))
    return valid_metrics, test_metrics


def evaluate_embedding_family(name, model, dim, train_tokens, valid_tokens, test_tokens, params):
    train_X, train_cov = make_mean_embeddings(train_tokens, model, dim)
    valid_X, valid_cov = make_mean_embeddings(valid_tokens, model, dim)
    test_X, test_cov = make_mean_embeddings(test_tokens, model, dim)
    valid_metrics, test_metrics = fit_eval_dense(
        train_X,
        train_df["topic"],
        valid_X,
        valid_df["topic"],
        test_X,
        test_df["topic"],
        params,
    )
    coverage_rows = [
        {"model": name, "split": "train", **train_cov},
        {"model": name, "split": "valid", **valid_cov},
        {"model": name, "split": "test", **test_cov},
    ]
    metric_rows = [
        {"model": name, "stage": "valid", **valid_metrics},
        {"model": name, "stage": "test", **test_metrics},
    ]
    del train_X, valid_X, test_X
    return coverage_rows, metric_rows

Запускаю оценку для всех трёх наборов эмбеддингов:


In [14]:
coverage_rows = []
metric_rows = []

rows_cov, rows_metrics = evaluate_embedding_family(
    "Word2Vec mean",
    w2v_model.wv,
    w2v_model.vector_size,
    train_df["lemma_tokens"].tolist(),
    valid_df["lemma_tokens"].tolist(),
    test_df["lemma_tokens"].tolist(),
    W2V_LOGREG_PARAMS,
)
coverage_rows.extend(rows_cov)
metric_rows.extend(rows_metrics)

rows_cov, rows_metrics = evaluate_embedding_family(
    "Navec mean",
    navec,
    navec.pq.dim,
    train_df["simple_tokens"].tolist(),
    valid_df["simple_tokens"].tolist(),
    test_df["simple_tokens"].tolist(),
    NAVEC_LOGREG_PARAMS,
)
coverage_rows.extend(rows_cov)
metric_rows.extend(rows_metrics)

rows_cov, rows_metrics = evaluate_embedding_family(
    "RusVectores mean",
    rusvectores,
    rusvectores.vector_size,
    train_df["rv_tokens"].tolist(),
    valid_df["rv_tokens"].tolist(),
    test_df["rv_tokens"].tolist(),
    RUSVECTORES_LOGREG_PARAMS,
)
coverage_rows.extend(rows_cov)
metric_rows.extend(rows_metrics)

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

,model,split,token_coverage,document_coverage
0,Word2Vec mean,train,0.984800,1.0
1,Word2Vec mean,valid,0.979910,1.0
2,Word2Vec mean,test,0.979834,1.0
3,Navec mean,train,0.981403,1.0
4,Navec mean,valid,0.981405,1.0
5,Navec mean,test,0.981282,1.0
6,RusVectores mean,train,0.698926,1.0
7,RusVectores mean,valid,0.698827,1.0
8,RusVectores mean,test,0.698788,1.0


Видим, что `token_coverage` < 1. Это потому что часть токенов нету в словарях моделей.

Метрики на validation и test:


In [15]:
comparison_df = pd.DataFrame(metric_rows)
display(comparison_df)

base_valid_df = comparison_df.query("stage == 'valid'").sort_values(["macro_f1", "accuracy"], ascending=False)
best_base_model = base_valid_df.iloc[0]["model"]
best_base_model

,model,stage,accuracy,macro_f1,weighted_f1
0,Word2Vec mean,valid,0.80015,0.604085,0.795709
1,Word2Vec mean,test,0.80155,0.601528,0.797476
2,Navec mean,valid,0.73840,0.576490,0.752098
3,Navec mean,test,0.74105,0.594580,0.754878
4,RusVectores mean,valid,0.76410,0.556766,0.758560
5,RusVectores mean,test,0.76105,0.585218,0.754937


'Word2Vec mean'

На validation лидирует `Word2Vec mean`: `accuracy = 0.800`, `macro F1 = 0.604`. Для сравнения, `Navec` показал `macro F1 = 0.576`, а `RusVectores` `0.557`.

Покрытием словаря эту разницу не объяснить. `Navec` покрывает ~98.1% токенов (даже чуть больше, чем Word2Vec с ~98.0%), но по метрикам уступает. Значит, дело в том, насколько само пространство эмбеддингов подходит к тематике `Lenta.ru`.

`RusVectores` дополнительно теряет в покрытии из-за формата `lemma_POS`: токенов найдено только ~70%.

На шаг с tf-idf взвешиванием беру `Word2Vec`, потому что он выиграл на validation.

## 5. Улучшение лучшего варианта через tf-idf взвешивание

Простое среднее считает все слова одинаково важными. Общие формулы и частотные служебные слова влияют на вектор документа примерно так же, как редкие тематические маркеры.

Пробую взвешенное среднее, где вес каждого слова берётся из tf-idf. Обучаю `TfidfVectorizer` на токенах, строю матрицу эмбеддингов для словаря vectorizer и умножаю sparse tf-idf матрицу на неё. Потом делю на сумму весов тех токенов, для которых эмбеддинг нашёлся.


Параметры tf-idf взвешивания:


In [16]:
TFIDF_WEIGHTED_PARAMS = {"min_df": 5, "max_df": 0.95, "sublinear_tf": False}

`min_df=5` игнорируем слова, которые встречаются менее чем в 5 разных документах.

`max_df=0.95` игнорируем слова, которые есть в более чем 95% документов.

`sublinear_tf=False` оставляет сырые частоты без логарифмирования.


Построение взвешенных эмбеддингов и сравнение с базовым средним:


In [ ]:
def build_tfidf_weighted_embeddings(
    token_lists_train: list[list[str]],
    token_lists_other: list[list[str]],
    model,
    dim: int,
    *,
    min_df: int,
    max_df: float,
    sublinear_tf: bool,
):
    vectorizer = TfidfVectorizer(
        analyzer=lambda doc: doc,
        lowercase=False,
        token_pattern=None,
        preprocessor=None,
        tokenizer=None,
        norm=None,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=sublinear_tf,
    )
    X_train = vectorizer.fit_transform(token_lists_train)
    X_other = vectorizer.transform(token_lists_other)

    vocab_items = sorted(vectorizer.vocabulary_.items(), key=lambda item: item[1])
    embedding_matrix = np.zeros((len(vocab_items), dim), dtype=np.float32)
    presence = np.zeros(len(vocab_items), dtype=np.float32)

    for token, index in vocab_items:
        if token in model:
            embedding_matrix[index] = model[token]
            presence[index] = 1.0

    def transform(matrix):
        weighted_sum = matrix @ embedding_matrix
        weight_denom = np.asarray(matrix @ presence).reshape(-1, 1)
        doc_vectors = np.zeros_like(weighted_sum, dtype=np.float32)
        np.divide(weighted_sum, weight_denom, out=doc_vectors, where=weight_denom > 0)
        doc_vectors[~np.isfinite(doc_vectors)] = 0.0
        return doc_vectors

    return transform(X_train), transform(X_other)


weighted_model = w2v_model.wv
weighted_dim = w2v_model.vector_size
weighted_train_tokens = train_df["lemma_tokens"].tolist()
weighted_valid_tokens = valid_df["lemma_tokens"].tolist()
weighted_test_tokens = test_df["lemma_tokens"].tolist()
weighted_lr_params = W2V_LOGREG_PARAMS

train_weighted_X, valid_weighted_X = build_tfidf_weighted_embeddings(
    weighted_train_tokens,
    weighted_valid_tokens,
    weighted_model,
    weighted_dim,
    **TFIDF_WEIGHTED_PARAMS,
)
train_valid_weighted_X, test_weighted_X = build_tfidf_weighted_embeddings(
    weighted_train_tokens + weighted_valid_tokens,
    weighted_test_tokens,
    weighted_model,
    weighted_dim,
    **TFIDF_WEIGHTED_PARAMS,
)

weighted_valid_model = build_dense_pipeline(**weighted_lr_params)
weighted_valid_model.fit(train_weighted_X, train_df["topic"])
weighted_valid_metrics = compute_metrics(valid_df["topic"], weighted_valid_model.predict(valid_weighted_X))

weighted_test_model = build_dense_pipeline(**weighted_lr_params)
weighted_test_model.fit(train_valid_weighted_X, pd.concat([train_df["topic"], valid_df["topic"]], ignore_index=True))
weighted_test_metrics = compute_metrics(test_df["topic"], weighted_test_model.predict(test_weighted_X))

weighted_summary_df = pd.DataFrame(
    [
        {
            "model": best_base_model,
            "stage": "valid",
            **base_valid_df.iloc[0][["accuracy", "macro_f1", "weighted_f1"]].to_dict(),
        },
        {"model": "TF-IDF weighted", "stage": "valid", **weighted_valid_metrics},
        {
            "model": best_base_model,
            "stage": "test",
            **comparison_df.query("stage == 'test' and model == @best_base_model")
            .iloc[0][["accuracy", "macro_f1", "weighted_f1"]]
            .to_dict(),
        },
        {"model": "TF-IDF weighted", "stage": "test", **weighted_test_metrics},
    ]
)
display(weighted_summary_df)

,model,stage,accuracy,macro_f1,weighted_f1
0,Word2Vec mean,valid,0.80015,0.604085,0.795709
1,TF-IDF weighted,valid,0.79625,0.623406,0.791523
2,Word2Vec mean,test,0.80155,0.601528,0.797476
3,TF-IDF weighted,test,0.79780,0.621378,0.793149


TF-IDF взвешивание подняло `macro F1` на validation с `0.604` до `0.623`, и на test с `0.602` до `0.621`. При этом accuracy чуть просела. Раз основная метрика `macro F1`, считаю tf-idf взвешенный вариант лучше простого среднего.

## 6. Финальное сравнение всех моделей на тестовой выборке

У меня четыре кандидата: три базовых варианта (`Word2Vec`, `Navec`, `RusVectores`) и tf-idf взвешенная версия лучшего. Модель выбираю по validation, переобучаю на `train + valid`, и потом смотрю на test.


In [18]:
final_test_df = pd.concat(
    [
        comparison_df.query("stage == 'test'"),
        pd.DataFrame([{"model": "TF-IDF weighted", "stage": "test", **weighted_test_metrics}]),
    ],
    ignore_index=True,
).sort_values(["macro_f1", "accuracy"], ascending=False)

display(final_test_df)

,model,stage,accuracy,macro_f1,weighted_f1
3,TF-IDF weighted,test,0.79780,0.621378,0.793149
0,Word2Vec mean,test,0.80155,0.601528,0.797476
1,Navec mean,test,0.74105,0.594580,0.754878
2,RusVectores mean,test,0.76105,0.585218,0.754937


## Выводы

На test лучше всех по `macro F1` оказался `TF-IDF weighted` вариант на основе `Word2Vec` (`0.621`). Простое среднее `Word2Vec` показало `macro F1 = 0.602` при лучшей accuracy (`0.802`). `Navec` занял третье место (`macro F1 = 0.595`), а `RusVectores` последнее (`0.585`).

TF-IDF взвешивание помогло и на validation (`macro F1` с `0.604` до `0.623`), и на test (`0.602` до `0.621`). Accuracy при этом чуть просела (`0.802` до `0.798`), но раз выбираем по `macro F1`, это приемлемый размен: модель стала лучше классифицировать редкие темы за счёт небольшой потери на частых.

`Navec` даёт почти полное покрытие (~98%) и хорош как готовый baseline. `RusVectores` при покрытии ~70% уступает остальным. Собственный `Word2Vec` оказался лучшей базой, потому что его пространство подстроилось под лексику `Lenta.ru`.